In [3]:

# ============================================================
# NB_Bronze_To_Silver
# Microsoft Fabric - Insurance Medallion Architecture
#
# Source      : LH_Bronze
# Destination : LH_Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    current_timestamp
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_SCHEMA = "dbo"
SILVER_LAKEHOUSE = "LH_Silver"
SILVER_SCHEMA = "dbo"


# ============================================================
# 2. READ BRONZE TABLES
# ============================================================

print("==============================================")
print("READING BRONZE TABLES")
print("==============================================")

customers_df = spark.table(
    f"{BRONZE_SCHEMA}.bronze_customers"
)

policies_df = spark.table(
    f"{BRONZE_SCHEMA}.bronze_policies"
)

claims_df = spark.table(
    f"{BRONZE_SCHEMA}.bronze_claims"
)

payments_df = spark.table(
    f"{BRONZE_SCHEMA}.bronze_payments"
)

print("Bronze tables loaded successfully.")

print(f"Bronze Customers : {customers_df.count()}")
print(f"Bronze Policies  : {policies_df.count()}")
print(f"Bronze Claims    : {claims_df.count()}")
print(f"Bronze Payments  : {payments_df.count()}")

# Helpful schema check
print("\nCUSTOMERS COLUMNS:")
print(customers_df.columns)

print("\nPOLICIES COLUMNS:")
print(policies_df.columns)

print("\nCLAIMS COLUMNS:")
print(claims_df.columns)

print("\nPAYMENTS COLUMNS:")
print(payments_df.columns)


# ============================================================
# 3. CUSTOMERS - BRONZE TO SILVER
# ============================================================

print("\nTransforming Customers...")

customers_silver = (
    customers_df

    .dropDuplicates(["customer_id"])

    .filter(
        col("customer_id").isNotNull()
    )

    .withColumn(
        "first_name",
        trim(col("first_name"))
    )

    .withColumn(
        "last_name",
        trim(col("last_name"))
    )

    .withColumn(
        "email",
        lower(trim(col("email")))
    )

    .withColumn(
        "state",
        upper(trim(col("state")))
    )

    .withColumn(
        "date_of_birth",
        to_date(
            col("date_of_birth"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "created_date",
        to_date(
            col("created_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "_silver_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 4. POLICIES - BRONZE TO SILVER
# ============================================================

print("Transforming Policies...")

policies_silver = (
    policies_df

    .dropDuplicates(["policy_id"])

    .filter(
        col("policy_id").isNotNull()
    )

    .withColumn(
        "product_type",
        upper(trim(col("product_type")))
    )

    .withColumn(
        "policy_status",
        upper(trim(col("policy_status")))
    )

    .withColumn(
        "policy_start_date",
        to_date(
            col("policy_start_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "policy_end_date",
        to_date(
            col("policy_end_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "annual_premium",
        col("annual_premium").cast("double")
    )

    .withColumn(
        "coverage_limit",
        col("coverage_limit").cast("double")
    )

    .withColumn(
        "deductible",
        col("deductible").cast("double")
    )

    .withColumn(
        "_silver_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 5. CLAIMS - BRONZE TO SILVER
# ============================================================

print("Transforming Claims...")

claims_silver = (
    claims_df

    .dropDuplicates(["claim_id"])

    .filter(
        col("claim_id").isNotNull()
    )

    .withColumn(
        "claim_type",
        upper(trim(col("claim_type")))
    )

    .withColumn(
        "claim_status",
        upper(trim(col("claim_status")))
    )

    .withColumn(
        "claim_date",
        to_date(
            col("claim_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "incident_date",
        to_date(
            col("incident_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "claim_amount",
        col("claim_amount").cast("double")
    )

    .withColumn(
        "approved_amount",
        col("approved_amount").cast("double")
    )

    .withColumn(
        "_silver_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 6. PAYMENTS - BRONZE TO SILVER
# ============================================================

print("Transforming Payments...")

payments_silver = (
    payments_df

    .dropDuplicates(["payment_id"])

    .filter(
        col("payment_id").isNotNull()
    )

    .withColumn(
        "payment_status",
        upper(trim(col("payment_status")))
    )

    .withColumn(
        "payment_date",
        to_date(
            col("payment_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "payment_amount",
        col("payment_amount").cast("double")
    )

    .withColumn(
        "_silver_processed_ts",
        current_timestamp()
    )
)


# ============================================================
# 7. GENERIC CREATE / REPLACE FUNCTION
# ============================================================

def create_or_replace_silver_table(
    dataframe,
    table_name
):

    full_table_name = (
        f"{SILVER_LAKEHOUSE}."
        f"{SILVER_SCHEMA}."
        f"{table_name}"
    )

    print(f"Creating/Replacing: {full_table_name}")

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"SUCCESS: {full_table_name}")


# ============================================================
# 8. WRITE ALL SILVER TABLES
# ============================================================

print("\n==============================================")
print("WRITING SILVER TABLES")
print("==============================================")

create_or_replace_silver_table(
    customers_silver,
    "silver_customers"
)

create_or_replace_silver_table(
    policies_silver,
    "silver_policies"
)

create_or_replace_silver_table(
    claims_silver,
    "silver_claims"
)

create_or_replace_silver_table(
    payments_silver,
    "silver_payments"
)


# ============================================================
# 9. READ SILVER TABLES BACK
# ============================================================

print("\n==============================================")
print("VALIDATING SILVER TABLES")
print("==============================================")

silver_customers_check = spark.table(
    "LH_Silver.dbo.silver_customers"
)

silver_policies_check = spark.table(
    "LH_Silver.dbo.silver_policies"
)

silver_claims_check = spark.table(
    "LH_Silver.dbo.silver_claims"
)

silver_payments_check = spark.table(
    "LH_Silver.dbo.silver_payments"
)


# ============================================================
# 10. ROW COUNT VALIDATION
# ============================================================

bronze_counts = {
    "customers": customers_df.count(),
    "policies": policies_df.count(),
    "claims": claims_df.count(),
    "payments": payments_df.count()
}

silver_counts = {
    "customers": silver_customers_check.count(),
    "policies": silver_policies_check.count(),
    "claims": silver_claims_check.count(),
    "payments": silver_payments_check.count()
}

print("\n----------------------------------------------")
print("BRONZE -> SILVER ROW COUNTS")
print("----------------------------------------------")

print(
    f"Customers : {bronze_counts['customers']} "
    f"-> {silver_counts['customers']}"
)

print(
    f"Policies  : {bronze_counts['policies']} "
    f"-> {silver_counts['policies']}"
)

print(
    f"Claims    : {bronze_counts['claims']} "
    f"-> {silver_counts['claims']}"
)

print(
    f"Payments  : {bronze_counts['payments']} "
    f"-> {silver_counts['payments']}"
)


# ============================================================
# 11. SAMPLE SILVER DATA
# ============================================================

print("\nSample Silver Claims:")

display(
    silver_claims_check.limit(10)
)


# ============================================================
# 12. COMPLETION
# ============================================================

print("\n==============================================")
print("BRONZE -> SILVER PROCESS COMPLETED")
print("==============================================")

print("Created/Replaced:")
print("LH_Silver.dbo.silver_customers")
print("LH_Silver.dbo.silver_policies")
print("LH_Silver.dbo.silver_claims")
print("LH_Silver.dbo.silver_payments")

print("==============================================")

StatementMeta(, c86b5783-7aba-4af9-bd53-e4e5d80b188d, 5, Finished, Available, Finished, False)

READING BRONZE TABLES
Bronze tables loaded successfully.
Bronze Customers : 501
Bronze Policies  : 751
Bronze Claims    : 1201
Bronze Payments  : 1501

CUSTOMERS COLUMNS:
['customer_id', 'first_name', 'last_name', 'date_of_birth', 'email', 'phone', 'address', 'city', 'state', 'zip_code', 'created_date', 'customer_status']

POLICIES COLUMNS:
['policy_id', 'customer_id', 'product_type', 'policy_start_date', 'policy_end_date', 'annual_premium', 'coverage_limit', 'deductible', 'policy_status', 'agent_id', 'last_updated']

CLAIMS COLUMNS:
['claim_id', 'policy_id', 'customer_id', 'claim_date', 'incident_date', 'claim_type', 'claim_amount', 'approved_amount', 'claim_status', 'description', 'reported_channel', 'adjuster_id', 'last_updated']

PAYMENTS COLUMNS:
['payment_id', 'policy_id', 'customer_id', 'claim_id', 'payment_date', 'payment_type', 'payment_amount', 'payment_method', 'payment_status', 'transaction_reference']

Transforming Customers...
Transforming Policies...
Transforming Claims.

SynapseWidget(Synapse.DataFrame, 762e1629-00b0-4708-97c5-ea49f23a7002)


BRONZE -> SILVER PROCESS COMPLETED
Created/Replaced:
LH_Silver.dbo.silver_customers
LH_Silver.dbo.silver_policies
LH_Silver.dbo.silver_claims
LH_Silver.dbo.silver_payments
